In [1]:
from pathlib import Path

import pandas as pd
import wbgapi as wb

## 1. World Bank country lookup table

In [2]:
world_bank_regions: pd.DataFrame = wb.economy.DataFrame()
world_bank_regions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 266 entries, ABW to ZWE
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         266 non-null    object 
 1   aggregate    266 non-null    bool   
 2   longitude    211 non-null    float64
 3   latitude     211 non-null    float64
 4   region       266 non-null    object 
 5   adminregion  266 non-null    object 
 6   lendingType  266 non-null    object 
 7   incomeLevel  266 non-null    object 
 8   capitalCity  266 non-null    object 
dtypes: bool(1), float64(2), object(6)
memory usage: 19.0+ KB


In [ ]:
world_bank_regions

,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
id,,,,,,,,,
ABW,Aruba,False,-70.0167,12.51670,LCN,,LNX,HIC,Oranjestad
AFE,Africa Eastern and Southern,True,NaN,NaN,,,,,
AFG,Afghanistan,False,69.1761,34.52280,MEA,MNA,IDX,LIC,Kabul
AFW,Africa Western and Central,True,NaN,NaN,,,,,
AGO,Angola,False,13.2420,-8.81155,SSF,SSA,IBD,LMC,Luanda
...,...,...,...,...,...,...,...,...,...
XKX,Kosovo,False,20.9260,42.56500,ECS,ECA,IDX,UMC,Pristina
YEM,"Yemen, Rep.",False,44.2075,15.35200,MEA,MNA,IDX,LIC,Sana'a
ZAF,South Africa,False,28.1871,-25.74600,SSF,SSA,IBD,UMC,Pretoria


In [ ]:
wb_regions_df = (
    world_bank_regions
    .reset_index()
    .rename(columns={'id': 'iso-alpha3_code'})
    .drop(columns=[
        'aggregate',
        'longitude',
        'latitude',
        'lendingType',
        'incomeLevel',
        'capitalCity'
    ])
    .replace({'': pd.NA})
)
wb_regions_df

,iso-alpha3_code,name,region,adminregion
0,ABW,Aruba,LCN,<NA>
1,AFE,Africa Eastern and Southern,<NA>,<NA>
2,AFG,Afghanistan,MEA,MNA
3,AFW,Africa Western and Central,<NA>,<NA>
4,AGO,Angola,SSF,SSA
...,...,...,...,...
261,XKX,Kosovo,ECS,ECA
262,YEM,"Yemen, Rep.",MEA,MNA
263,ZAF,South Africa,SSF,SSA
264,ZMB,Zambia,SSF,SSA


In [5]:
wb_regions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   iso-alpha3_code  266 non-null    object
 1   name             266 non-null    object
 2   region           217 non-null    object
 3   adminregion      129 non-null    object
dtypes: object(4)
memory usage: 8.4+ KB


## 2. UN M49 country lookup table

In [6]:
project_dir = Path.cwd().parent
common_table_dir = project_dir / 'data' / 'common-tables'
un_countries_path = common_table_dir / 'UNSD — Methodology.csv'

un_countries_df: pd.DataFrame = pd.read_csv(un_countries_path, sep=';')
original_un_countries_df = un_countries_df.copy(deep=True)

In [7]:
un_countries_df.head()

,Global Code,Global Name,Region Code,Region Name,Sub-region Code,Sub-region Name,Intermediate Region Code,Intermediate Region Name,Country or Area,M49 Code,ISO-alpha2 Code,ISO-alpha3 Code,Least Developed Countries (LDC),Land Locked Developing Countries (LLDC),Small Island Developing States (SIDS)
0,1,World,2.0,Africa,15.0,Northern Africa,NaN,NaN,Algeria,12,DZ,DZA,NaN,NaN,NaN
1,1,World,2.0,Africa,15.0,Northern Africa,NaN,NaN,Egypt,818,EG,EGY,NaN,NaN,NaN
2,1,World,2.0,Africa,15.0,Northern Africa,NaN,NaN,Libya,434,LY,LBY,NaN,NaN,NaN
3,1,World,2.0,Africa,15.0,Northern Africa,NaN,NaN,Morocco,504,MA,MAR,NaN,NaN,NaN
4,1,World,2.0,Africa,15.0,Northern Africa,NaN,NaN,Sudan,729,SD,SDN,x,NaN,NaN


In [8]:
un_countries_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 15 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Global Code                              248 non-null    int64  
 1   Global Name                              248 non-null    object 
 2   Region Code                              247 non-null    float64
 3   Region Name                              247 non-null    object 
 4   Sub-region Code                          247 non-null    float64
 5   Sub-region Name                          247 non-null    object 
 6   Intermediate Region Code                 105 non-null    float64
 7   Intermediate Region Name                 105 non-null    object 
 8   Country or Area                          248 non-null    object 
 9   M49 Code                                 248 non-null    int64  
 10  ISO-alpha2 Code                          247 non-n

In [9]:
un_region_codes_df = (
    un_countries_df
    .filter(
        items=['M49 Code', 'ISO-alpha2 Code', 'ISO-alpha3 Code'],
        axis='columns'
    )
    .rename(columns={'M49 Code': 'm49_code',
                     'ISO-alpha2 Code': 'iso-alpha2_code',
                     'ISO-alpha3 Code': 'iso-alpha3_code'})
)
un_region_codes_df.head(3)

,m49_code,iso-alpha2_code,iso-alpha3_code
0,12,DZ,DZA
1,818,EG,EGY
2,434,LY,LBY


In [ ]:
un_levels: list[tuple] = [
    ('Global Code', 'Global Name'),
    ('Region Code', 'Region Name'),
    ('Sub-region Code', 'Sub-region Name'),
    ('Intermediate Region Code', 'Intermediate Region Name'),
    ('M49 Code', 'Country or Area'),
]

level_frames: list[pd.DataFrame] = []
for code_col, name_col in un_levels:
    temp = (
        un_countries_df
        .dropna(subset=[code_col])
        [[code_col, name_col]]  # Remove all cols except Code & Name
        .rename(columns={code_col: 'm49_code', name_col: 'm49_name'})
    )
    level_frames.append(temp)

# Construct country lookup table
all_regions_df = (
    pd
    .concat(level_frames, ignore_index=True)
    .drop_duplicates(subset=['m49_code', 'm49_name'])
    .sort_values(by='m49_code')
    .reset_index(drop=True)
    .convert_dtypes()
)

# Merge m49 codes with ISO-alpha codes & World Bank names
all_regions_df = all_regions_df.merge(un_region_codes_df, on='m49_code', how='left')
all_regions_df = (
    all_regions_df
    .merge(wb_regions_df, on='iso-alpha3_code', how='outer')
    .rename(columns={'name': 'world_bank_name'})
    .drop(columns=['region', 'adminregion'])
)

In [38]:
all_regions_df.columns

Index(['m49_code', 'm49_name', 'iso-alpha2_code', 'iso-alpha3_code',
       'world_bank_name'],
      dtype='object')

### Import StatCan regions from `StatCan-transformations.ipynb`

In [30]:
%store -r statcan_regions

# Convert `statcan_regions` to a dataframe
statcan_regions_df: pd.DataFrame = statcan_regions.rename('statcan_name').to_frame()
original_statcan_regions_df = statcan_regions_df.copy(deep=True)
statcan_regions_df.head()

,statcan_name
0,Afghanistan
1,Africa
2,Albania
3,Algeria
4,All countries


In [12]:
un_merged = statcan_regions_df.merge(
    all_regions_df,
    how='left',
    left_on='statcan_name',
    right_on='m49_name',
)
wb_merged = statcan_regions_df.merge(
    all_regions_df,
    how='left',
    left_on='statcan_name',
    right_on='world_bank_name',
)

# Coalesce: combine where `un_merge` is Null
statcan_merged = (
    un_merged
    .combine_first(wb_merged)
    .convert_dtypes()
)
statcan_merged

,statcan_name,m49_code,m49_name,iso-alpha2_code,iso-alpha3_code,world_bank_name
0,Afghanistan,4,Afghanistan,AF,AFG,Afghanistan
1,Africa,2,Africa,<NA>,<NA>,<NA>
2,Albania,8,Albania,AL,ALB,Albania
3,Algeria,12,Algeria,DZ,DZA,Algeria
4,All countries,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...
265,Yemen,887,Yemen,YE,YEM,"Yemen, Rep."
266,Yugoslavia,<NA>,<NA>,<NA>,<NA>,<NA>
267,Zaire,<NA>,<NA>,<NA>,<NA>,<NA>
268,Zambia,894,Zambia,ZM,ZMB,Zambia


In [13]:
statcan_unmatched = (
    statcan_merged.loc[
        statcan_merged[['m49_name', 'world_bank_name']].isna().all(axis=1)
        & statcan_merged['statcan_name'].notna()
    ]
)
statcan_unmatched.head()

,statcan_name,m49_code,m49_name,iso-alpha2_code,iso-alpha3_code,world_bank_name
4,All countries,<NA>,<NA>,<NA>,<NA>,<NA>
15,Asia/Oceania,<NA>,<NA>,<NA>,<NA>,<NA>
40,Burma,<NA>,<NA>,<NA>,<NA>,<NA>
49,"Central America, South America, and Caribbean",<NA>,<NA>,<NA>,<NA>,<NA>
58,"Congo, Democratic Republic of the",<NA>,<NA>,<NA>,<NA>,<NA>


In [14]:
wb_names = set(all_regions_df['world_bank_name'].dropna())
un_names = set(all_regions_df['m49_name'].dropna())
statcan_unmatched_names = set(statcan_unmatched['statcan_name'].dropna())

to_be_matched = {
    'statcan_unmatched': sorted(list(statcan_unmatched_names)),
    'un_names': sorted(list(un_names)),
    'world_bank_names': sorted(list(wb_names))
}
to_be_matched

{'statcan_unmatched': ['All countries',
  'Asia/Oceania',
  'Burma',
  'Central America, South America, and Caribbean',
  'Congo, Democratic Republic of the',
  'Czechoslovakia',
  "Côte d'Ivoire",
  'Federated States of Micronesia',
  'German Democratic Republic (East)',
  'High Seas',
  'Hong Kong',
  'Iran',
  'Laos',
  'Macao',
  'Middle East',
  'Netherlands Antilles',
  'North Korea',
  'Oceania and Antarctica',
  'Other Africa',
  'Other Asia and Oceania',
  'Other Caribbean',
  'Other Europe',
  'Other South and Central America',
  'Pacific Islands',
  'Saint Martin (French part)',
  'South Korea',
  'South and Central America',
  'Syria',
  'Taiwan',
  'Tanzania, United Republic of',
  'Unallocated countries',
  'Union of Soviet Socialist Republics',
  'Unspecified',
  'Venezuela',
  'Wallis and Futuna',
  'Yugoslavia',
  'Zaire'],
 'un_names': ['Afghanistan',
  'Africa',
  'Albania',
  'Algeria',
  'American Samoa',
  'Americas',
  'Andorra',
  'Angola',
  'Anguilla',
  'Anta

In [ ]:
# Matches made by ChatGPT o3, then manually reviewed
# Ambiguous matches moved to 'possible matches'
matched_by_ai = {
    "matches": [
        { "statcan": "All countries", "un": "World", "world_bank": "World" },
        { "statcan": "Burma", "un": "Myanmar", "world_bank": "Myanmar" },
        { "statcan": "Central America, South America, and Caribbean", "un": "Latin America and the Caribbean", "world_bank": "Latin America & Caribbean" },
        { "statcan": "Congo, Democratic Republic of the", "un": "Democratic Republic of the Congo", "world_bank": "Congo, Dem. Rep." },
        { "statcan": "Côte d'Ivoire", "un": "Côte d’Ivoire", "world_bank": "Cote d'Ivoire" },
        { "statcan": "Federated States of Micronesia", "un": "Micronesia (Federated States of)", "world_bank": "Micronesia, Fed. Sts." },
        { "statcan": "Hong Kong", "un": "China, Hong Kong Special Administrative Region", "world_bank": "Hong Kong SAR, China" },
        { "statcan": "Iran", "un": "Iran (Islamic Republic of)", "world_bank": "Iran, Islamic Rep." },
        { "statcan": "Laos", "un": "Lao People's Democratic Republic", "world_bank": "Lao PDR" },
        { "statcan": "Macao", "un": "China, Macao Special Administrative Region", "world_bank": "Macao SAR, China" },
        { "statcan": "Netherlands Antilles", "un": "Curaçao", "world_bank": "Curacao" },
        { "statcan": "North Korea", "un": "Democratic People's Republic of Korea", "world_bank": "Korea, Dem. People's Rep." },
        { "statcan": "Saint Martin (French part)", "un": "Saint Martin (French Part)", "world_bank": "St. Martin (French part)" },
        { "statcan": "South Korea", "un": "Republic of Korea", "world_bank": "Korea, Rep." },
        { "statcan": "Syria", "un": "Syrian Arab Republic", "world_bank": "Syrian Arab Republic" },
        { "statcan": "Tanzania, United Republic of", "un": "United Republic of Tanzania", "world_bank": "Tanzania" },
        { "statcan": "Venezuela", "un": "Venezuela (Bolivarian Republic of)", "world_bank": "Venezuela, RB" },
        { "statcan": "Wallis and Futuna", "un": "Wallis and Futuna Islands" },
        { "statcan": "Zair", "un": "Democratic Republic of the Congo", "world_bank": "Congo, Dem. Rep." },
    ],
    "possible_matches": [
        { "statcan": "Middle East", "un": "Western Asia", "world_bank": "Middle East, North Africa, Afghanistan & Pakistan" },
        { "statcan": "Pacific Islands", "un": "Oceania", "world_bank": "Pacific island small states" },
        { "statcan": "South and Central America", "un": "South America", "world_bank": "Latin America & Caribbean" },
        { "statcan": "Other Caribbean", "un": "Caribbean", "world_bank": "Caribbean small states" },
        { "statcan": "Other Europe", "un": "Europe", "world_bank": "Europe & Central Asia" },
        { "statcan": "Other South and Central America", "un": "South America", "world_bank": "Latin America & Caribbean" }
    ],
    "statcan_only": [
        "Asia/Oceania",
        "Czechoslovakia",
        "German Democratic Republic (East)",
        "High Seas",
        "Oceania and Antarctica",
        "Other Africa",
        "Other Asia and Oceania",
        "Unallocated countries",
        "Union of Soviet Socialist Republics",
        "Unspecified",
        "Taiwan",
        "Yugoslavia"
    ]
}

# Create df of matches
un_matches_by_ai = []
statcan_matched_by_ai = []
for match in matched_by_ai['matches']:
    for key, value in match.items():
        if key == 'statcan':
            statcan_matched_by_ai.append(value)
        elif key == 'un':
            un_matches_by_ai.append(value)
  
matched_by_ai_df = pd.DataFrame({'statcan_name': statcan_matched_by_ai,
                                 'm49_name': un_matches_by_ai})

# Create df of StatCan-exclusive regions
statcan_exclusive = []
for possible_match in matched_by_ai['possible_matches']:
    for key, value in possible_match.items():
        if key == 'statcan':
            statcan_exclusive.append(value)

statcan_exclusive_df = pd.DataFrame({'statcan_name': statcan_exclusive})

In [43]:
# statcan_matched_names = list(
#     set(statcan_regions_df['statcan_name'].to_list())
#     - set(statcan_matched_by_ai)
# )
# statcan_matched_df = pd.DataFrame({'statcan_name': statcan_matched_names})

KeyError: "None of ['m49_name'] are in the columns"

In [ ]:
# # Insert surrogate key once all transformations are complete
# all_regions_df.insert(0, 'region_id', all_regions_df.index+1)

# # Finally, reorder columns to final positions
# all_codes_df_col_order = [
#     'region_id',
#     'iso-alpha3_code',
#     'iso-alpha2_code',
#     'm49_code',
#     'm49_name',
#     'world_bank_name',
#     'statcan_name
# ]
# all_regions_df = all_regions_df[all_codes_df_col_order]

# all_regions_df.head()

### Aside: UN and World Bank have mutually exclusive ISO country codes

In [136]:
un_exclusive_iso_codes = (
    set(un_region_codes_df['iso-alpha3_code']) - set(wb_regions_df['iso-alpha3_code'])
)
wb_exclusive_iso_codes = (
    set(wb_regions_df['iso-alpha3_code']) - set(un_region_codes_df['iso-alpha3_code'])
)

exclusive_iso_codes = {
    'code_type': 'iso-alpha3_code',
    'wb_exclusive_codes': sorted(list(wb_exclusive_iso_codes)),
    'un_exclusive_codes': sorted(list(un_exclusive_iso_codes))
}

print(f'UN-exclusive ISO-Alpha 3 codes: {len(un_exclusive_iso_codes)}')
print(f'World Bank-exclusive ISO-Alpha 3 codes: {len(wb_exclusive_iso_codes)}')

UN-exclusive ISO-Alpha 3 codes: 33
World Bank-exclusive ISO-Alpha 3 codes: 51


In [ ]:


print(f'    World Bank name count: {len(wb_names)}\n'
      f'United Nations name count: {len(un_names)}')

wb_exclusive_names = wb_names - un_names
un_exclusive_names = un_names - wb_names

exclusive_names = {
    'wb_exclusive_names': sorted(list(wb_exclusive_names)),
    'un_exclusive_names': sorted(list(un_exclusive_names))
}
exclusive_names

    World Bank name count: 266
United Nations name count: 278
World Bank exclusive names:
    Heavily indebted poor countries (HIPC)
    Korea, Rep.
    IDA total
    Virgin Islands (U.S.)
    Kyrgyz Republic
    IBRD only
    Netherlands
    Macao SAR, China
    Curacao
    OECD members
    St. Kitts and Nevis
    Low & middle income
    Moldova
    North America
    Africa Eastern and Southern
    Pacific island small states
    Cote d'Ivoire
    Slovak Republic
    South Asia
    Lower middle income
    Pre-demographic dividend
    Middle income
    European Union
    Middle East, North Africa, Afghanistan & Pakistan (excluding high income)
    West Bank and Gaza
    United Kingdom
    Caribbean small states
    Early-demographic dividend
    East Asia & Pacific
    St. Vincent and the Grenadines
    Gambia, The
    Lao PDR
    Africa Western and Central
    Congo, Rep.
    Upper middle income
    St. Martin (French part)
    Channel Islands
    Central Europe and the Baltics
    Fr